# exp09 — probe score study

Four questions, four sections, nothing else.

| # | question | table |
|---|---|---|
| 1 | For each cell and each v1 task, which pooling scores higher — `mean` or `mean_std`? | absolute PR-AUC, better one highlighted |
| 2 | Within a cell, does validation reconstruction loss predict probe score across seeds? | Spearman rho over that cell's seeds |
| 3 | On all 10 reportable tasks, how do **four cells'** µ stand against the engineered features under a **nonlinear** readout? | C3b absolutes for 4 cells + fusion deltas, linear / GBM / MLP |
| 4 | On the same 10 tasks, how do three cells' µ stand against a model trained **with labels**? | absolutes for `features` / `µ` / `features ⊕ µ` / **C1** / **C2**, one table per cell |

**Probe score means `mean` pooling throughout** (decision 2026-08-29). Section 1 measures the two
poolings against each other because that is its question; sections 2–4 report `mean` and nothing
else. `max(mean, mean_std)` per task is **not** used anywhere — it would pick the readout on the test
score.

Sections 1–2 are about **exp09 cells** (the aux-loss amount axis, v1 tasks only, µ-only readout).
Section 3 takes **four** of those cells — the incumbent `hann w=0.30` plus the top three on `mean4` —
and measures them against the 25 engineered features on the full 10-task scorecard. Section 4 keeps that
10-task frame and adds the two **supervised** arms (roadmap C1/C2), for three cells. Different
questions, different data sources, and they are kept downstream of sections 1–2 rather than mixed into
them.

**Cells** (sections 1–2) = every exp09 cell that varies the *aux-loss amount* on the frozen w256/seq16
geometry, in four groups:

| group | aux window | kurtosis penalty | free bits | arm | cells |
|---|---|---|---|---|---|
| `hann` | Hann | 0 | 0 | fwd_bwd | w = 0.30 (the incumbent), 0.10, 0.025 |
| `dpss_impulse` | DPSS | 0.1 | 0 | fwd_bwd | w = 0.30 … 0.0125, 8 weights |
| `dpss_impulse … fb0.01` | DPSS | 0.1 | **0.01** | fwd_bwd | w = 0.03, 0.025 |
| `dpss_impulse … dyn-off` | DPSS | 0.1 | 0 | **dyn-off** (`lambda_dyn=0`) | w = 0.025 |

The last row is an **arm control, not a loss variant**. exp06 established that every geometry number is
arm-specific, so read it against the other rows only as a dyn-on/dyn-off contrast — never average it in.

**Section 3 uses four of these cells**: `hann w=0.30`, `hann w=0.10`, `dpss w=0.03 fb0.01`,
`dpss w=0.025 fb0.01`. **Section 4 uses three**: `hann w=0.30`, `dpss w=0.03 fb0.01`,
`dpss w=0.025 fb0.01`. The two sections read the `fb0.01` cells at **different checkpoints** — section 3
at `best_recon_aux` (the only one its incumbent has), section 4 at `best_recon_only` (A3) — and section
3 measures the resulting disagreement rather than assuming it away: max **0.0030** over the 40 rows the
two share.

The mechanism cells (`aux_none`, `aux_clip`, `aux_dpss`, `aux_impulse_pen`) and the hinge-floor cells
(`clip_hinge_*`) are **out of scope here**: they change *what* the aux loss is, not how much of it, and
sections 1–2 are about the amount axis.

**This notebook deliberately shows nothing else** — no gates, no verdicts, no encoder decision. In
particular sections 3 and 4 are **not** a re-opening of D17: the encoder decision closed 2026-08-26 on
`hann0p3`, and these tables carry no switch rule.

In [1]:
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy.stats import spearmanr

ROOT = Path.cwd()
while not (ROOT / "experiments").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
EXP = ROOT / "experiments"
CURVES = EXP / "exp09_forensics" / "curves_exp09"
CFGS = ROOT / "src" / "swm" / "configs" / "experiment" / "exp09"

TASKS = ["pulsating", "eb", "rotation", "transit"]
POOLS = ["mean", "mean_std"]
# display label -> experiment cell, grouped by family, aux weight descending inside each group.
CELLS = {
    "hann  w=0.30 (incumbent)": "exp07_hann0p3_fbwd",
    "hann  w=0.10": "exp09_hann_w0p10",
    "hann  w=0.025": "exp09_hann_w0p025",
    "dpss_impulse  w=0.30": "exp09_aux_dpss_impulse",
    "dpss_impulse  w=0.20": "exp09_dpss_impulse_w0p20",
    "dpss_impulse  w=0.10": "exp09_dpss_impulse_w0p10",
    "dpss_impulse  w=0.05": "exp09_dpss_impulse_w0p05",
    "dpss_impulse  w=0.03": "exp09_dpss_impulse_w0p03",
    "dpss_impulse  w=0.025": "exp09_dpss_impulse_w0p025",
    "dpss_impulse  w=0.02": "exp09_dpss_impulse_w0p02",
    "dpss_impulse  w=0.0125": "exp09_dpss_impulse_w0p0125",
    "dpss_impulse  w=0.03  fb0.01": "exp09_dpss_impulse_w0p03_fb0p01",
    "dpss_impulse  w=0.025  fb0.01": "exp09_dpss_impulse_w0p025_fb0p01",
    "dpss_impulse  w=0.025  dyn-off": "exp09_dpss_impulse_w0p025_off",
}

probe = (pd.concat([pd.read_csv(f) for f in glob.glob(str(EXP / "exp09_diag_*_probe_summary.csv"))],
                   ignore_index=True)
         .drop_duplicates(["cell", "seed", "pooling", "task"]))

# Footing check (clarity rule 8): re-derive a published number rather than trusting the load.
_eb6 = probe.query("cell == 'exp07_hann0p3_fbwd' and pooling == 'mean' and task == 'eb' and seed < 6")
assert abs(_eb6.pr_auc.mean() - 0.7710) < 5e-4, _eb6.pr_auc.mean()


def knobs(cell: str) -> dict:
    """Read the knobs that separate these cells out of the generated Hydra config, not from memory."""
    cfg = yaml.safe_load((CFGS / f"{cell}.yaml").read_text())
    aux, tr = cfg["train"]["recon_aux"], cfg["train"]
    return {"aux window": aux["psd_window"], "aux weight": aux["weight"],
            "kurtosis pen": tr["impulse_penalty_weight"], "free bits": tr["free_bits"],
            "arm": "dyn-off" if tr["lambda_dyn"] == 0 else cfg["model"]["dyn_mode"]}


seeds_of = {lab: sorted(probe.loc[probe.cell == c, "seed"].unique()) for lab, c in CELLS.items()}
cells_meta = pd.DataFrame({lab: {"cell": c, **knobs(c), "n_seeds": len(seeds_of[lab])}
                           for lab, c in CELLS.items()}).T
assert (cells_meta.n_seeds > 0).all(), cells_meta[cells_meta.n_seeds == 0]
cells_meta

,cell,aux window,aux weight,kurtosis pen,free bits,arm,n_seeds
hann w=0.30 (incumbent),exp07_hann0p3_fbwd,hann,0.3,0.0,0.0,fwd_bwd,12
hann w=0.10,exp09_hann_w0p10,hann,0.1,0.0,0.0,fwd_bwd,6
hann w=0.025,exp09_hann_w0p025,hann,0.025,0.0,0.0,fwd_bwd,6
dpss_impulse w=0.30,exp09_aux_dpss_impulse,dpss,0.3,0.1,0.0,fwd_bwd,6
dpss_impulse w=0.20,exp09_dpss_impulse_w0p20,dpss,0.2,0.1,0.0,fwd_bwd,6
dpss_impulse w=0.10,exp09_dpss_impulse_w0p10,dpss,0.1,0.1,0.0,fwd_bwd,6
dpss_impulse w=0.05,exp09_dpss_impulse_w0p05,dpss,0.05,0.1,0.0,fwd_bwd,6
dpss_impulse w=0.03,exp09_dpss_impulse_w0p03,dpss,0.03,0.1,0.0,fwd_bwd,6
dpss_impulse w=0.025,exp09_dpss_impulse_w0p025,dpss,0.025,0.1,0.0,fwd_bwd,12
dpss_impulse w=0.02,exp09_dpss_impulse_w0p02,dpss,0.02,0.1,0.0,fwd_bwd,6


## 1. Which pooling scores higher, per cell and per task?

Both poolings run the identical frozen-encoder linear probe; they differ only in what is fed to it.

```
  mean      star vector = mean over the star's window mu            ->  z dims
  mean_std  star vector = [ mean over windows | std over windows ]  ->  2z dims
```

The number in every cell is **absolute test PR-AUC, averaged over that cell's seeds** — not a delta
against anything. Higher is better.

```
  score(cell, task, pool) = mean over seeds of  PR-AUC(cell, seed, task, pool)
```

**Highlighted = the higher of the two poolings** for that (cell, task).

> **Decision, 2026-08-29 — the reported probe score is `mean`.** Not `mean_std`, and *not*
> `max(mean, mean_std)` per task. This table stays because it is the evidence behind that decision, but
> a green cell in the `mean_std` column is **not** a score this project quotes. Taking the per-task
> maximum would select the readout on the test score, which inflates every cell and inflates the noisiest
> cells most; N1/D2 already fix the ladder as `mean` headline, `mean_std` second, `mean ⊥ amp` appendix.
> Sections 2 and 3 report `mean` only.

Prevalence differs per task and PR-AUC is prevalence-dependent, so compare **down a column, never
across columns** (R8-F1).

In [2]:
rows = {}
for label, cell in CELLS.items():
    rows[label] = {(t, p): probe.query("cell == @cell and task == @t and pooling == @p").pr_auc.mean()
                   for t in TASKS for p in POOLS}
pooling_table = pd.DataFrame(rows).T
pooling_table.columns = pd.MultiIndex.from_tuples(pooling_table.columns, names=["task", "pooling"])
pooling_table = pooling_table[[(t, p) for t in TASKS for p in POOLS]]


def highlight_better_pooling(row: pd.Series) -> pd.Series:
    """Bold + green the higher-scoring pooling within each task's pair of columns."""
    style = pd.Series("", index=row.index)
    for task in TASKS:
        winner = (task, "mean") if row[(task, "mean")] > row[(task, "mean_std")] else (task, "mean_std")
        style[winner] = "font-weight: bold; background-color: #cdeccd"
    return style


pooling_table.style.apply(highlight_better_pooling, axis=1).format("{:.4f}")

### What this table shows, and what it does not

**Shows.** Which pooling wins is mostly a property of the **task**, not of the cell — but only for
three of the four tasks. Counts below are over the **13 dyn-on cells**; the dyn-off control is tallied
separately in the next cell because it is a different arm.

```
  rotation   mean      wins 13/13   median gap -0.038   <- unanimous, largest gap toward `mean`
  transit    mean_std  wins 13/13   median gap +0.064   <- unanimous, largest gap toward `mean_std`
  eb         mean      wins 12/13   median gap -0.011   <- 1 exception (the incumbent), 0.0014 wide
  pulsating  split      5 vs 8      median gap +0.002   <- NOT a task-level rule
```

`pulsating` is the honest exception: the two poolings sit within a few thousandths of each other in
almost every cell, so which one "wins" there is seed noise being ranked, not a pooling effect. Adding
the six new cells left the three clear tasks untouched and made pulsating's split more visible, not
less. The dyn-off control agrees with the dyn-on majority on rotation and transit, and disagrees on eb
and pulsating — the same two tasks whose gaps are near zero.

**Does not show.** Nothing here is a comparison *between* cells — no paired delta, no SE, no gate.
Two cells' absolute scores may differ by less than their seed-to-seed spread; this table cannot tell
you whether a difference is real. In particular, the distance between the dyn-off row and the dyn-on
rows is an arm contrast and is not scored anywhere in this notebook.

In [3]:
DYN_ON = [lab for lab in CELLS if cells_meta.loc[lab, "arm"] != "dyn-off"]  # tally the arm separately
DYN_OFF = [lab for lab in CELLS if cells_meta.loc[lab, "arm"] == "dyn-off"]
on, off = pooling_table.loc[DYN_ON], pooling_table.loc[DYN_OFF]

wins = pd.DataFrame(
    {task: {f"cells where `mean` wins (of {len(DYN_ON)} dyn-on)":
                int((on[(task, "mean")] > on[(task, "mean_std")]).sum()),
            f"cells where `mean_std` wins (of {len(DYN_ON)} dyn-on)":
                int((on[(task, "mean_std")] > on[(task, "mean")]).sum()),
            "median gap (mean_std - mean), dyn-on":
                float((on[(task, "mean_std")] - on[(task, "mean")]).median()),
            "dyn-off cell: winner":
                "mean_std" if (off[(task, "mean_std")] > off[(task, "mean")]).all() else "mean",
            "dyn-off cell: gap (mean_std - mean)":
                float((off[(task, "mean_std")] - off[(task, "mean")]).iloc[0])}
     for task in TASKS}).T
wins.index.name = "task"
wins.round(4)

,cells where `mean` wins (of 13 dyn-on),cells where `mean_std` wins (of 13 dyn-on),"median gap (mean_std - mean), dyn-on",dyn-off cell: winner,dyn-off cell: gap (mean_std - mean)
task,,,,,
pulsating,5,8,0.001722,mean_std,0.036094
eb,12,1,-0.011185,mean_std,0.006199
rotation,13,0,-0.037593,mean,-0.023571
transit,0,13,0.063909,mean_std,0.045806


## 2. Does reconstruction loss predict probe score within a cell?

This is the model-selection question: given several seeds of **one** recipe, would picking the seed
with the best validation reconstruction pick a good representation?

```
  recon(seed)  = min over epochs of  val/recon      # a LOSS: lower is better
  rho(cell, task) = Spearman( recon(seed) , PR-AUC(seed) )  over that cell's seeds
```

**Probe score = `mean` pooling only** (decision above). `mean_std` is dropped from here down: the rho
would be a correlation against a number the project does not report, and computing it for both would
double the multiple-comparison burden of a section whose whole point is that its highlights are near the
chance rate. Flip `REPORTED` in the next cell to get the `mean_std` version back.

**Sign convention — read it before the numbers:**

```
  rho < 0  ->  lower recon loss goes with HIGHER probe  ->  recon loss IS useful for selection
  rho > 0  ->  lower recon loss goes with LOWER  probe  ->  recon loss is ANTI-predictive
  rho ~ 0  ->  recon loss carries no information about probe quality
```

`mean4` = the same rho computed against the unweighted average of the four task PR-AUCs.

**Decision rule, stated before the result.** A Spearman rho is significant at p < 0.05 (two-sided)
when `|rho| >= 0.829` at n = 6 and `|rho| >= 0.587` at n = 12. Cells reaching their own threshold are
highlighted. With 14 cells x 5 columns = 70 tests, **about 3.5 false positives are expected by chance**,
so an isolated highlight is not evidence.

Note on the two floored cells: `free_bits = 0.01` puts a floor under the KL term, which changes what
`val/recon` is trading against. Their rho is still a within-cell seed ranking and is computed the same
way, but it is not the same quantity as an unfloored cell's — do not read the floored and unfloored
rows as one series.

In [4]:
CRIT = {6: 0.829, 12: 0.587}  # |rho| for p < 0.05 two-sided, Spearman exact
REPORTED = ["mean"]           # the reported probe score; set to POOLS to restore the mean_std rows

rows = []
for label, cell in CELLS.items():
    seeds = seeds_of[label]
    recon = [pd.read_csv(CURVES / f"{cell}_B_seed{s}.csv")["val/recon"].min() for s in seeds]
    for pool in REPORTED:
        per_task = {t: probe.query("cell == @cell and task == @t and pooling == @pool")
                    .set_index("seed").pr_auc.reindex(seeds) for t in TASKS}
        row = {"cell": label, "pooling": pool, "n": len(seeds)}
        for t in TASKS:
            row[t] = spearmanr(recon, per_task[t].values).statistic
        row["mean4"] = spearmanr(recon, sum(per_task.values()).values / 4).statistic
        rows.append(row)
rho_table = pd.DataFrame(rows).set_index(["cell", "pooling"])


def flag_significant(row: pd.Series) -> pd.Series:
    """Highlight rho values reaching that row's own n-dependent significance threshold."""
    style = pd.Series("", index=row.index)
    for col in TASKS + ["mean4"]:
        if abs(row[col]) >= CRIT[int(row["n"])]:
            style[col] = "font-weight: bold; background-color: #f3d4d4"
    return style


rho_table.style.apply(flag_significant, axis=1).format({c: "{:+.3f}" for c in TASKS + ["mean4"]})

,,n,pulsating,eb,rotation,transit,mean4
cell,pooling,,,,,,
hann w=0.30 (incumbent),mean,12,+0.147,-0.301,-0.175,+0.084,+0.000
hann w=0.10,mean,6,-0.314,-0.257,+0.886,+0.371,+0.200
hann w=0.025,mean,6,+0.486,-0.829,-0.257,+0.029,+0.143
dpss_impulse w=0.30,mean,6,+0.771,-0.086,-0.257,+0.543,+0.600
dpss_impulse w=0.20,mean,6,-0.200,-0.314,-0.657,+0.429,-0.600
dpss_impulse w=0.10,mean,6,+0.600,+0.029,+0.429,-0.029,+0.771
dpss_impulse w=0.05,mean,6,-0.200,+0.257,-0.200,-0.543,-0.257
dpss_impulse w=0.03,mean,6,-0.314,+0.143,+0.143,-0.257,+0.086
dpss_impulse w=0.025,mean,12,+0.063,-0.049,-0.119,-0.832,-0.014


### What this table shows, and what it does not

**Shows.** The magnitude and the sign consistency of the recon-loss / probe relationship inside each
recipe. The summary below gives the mean `|rho|` and how often the sign is the useful one.

**Does not show.** It cannot separate "recon loss is uninformative" from "seeds of one recipe are too
alike for any statistic to rank them" — the spread being ranked is seed noise within a fixed recipe,
not a spread of genuinely different models. It also says nothing about ranking *across* recipes, which
is a different question with a different null.

In [5]:
cols = TASKS + ["mean4"]
summary = pd.DataFrame({
    "mean |rho|": rho_table[cols].abs().mean(axis=1),
    "max |rho|": rho_table[cols].abs().max(axis=1),
    "n_significant / 5": [int(sum(abs(r[c]) >= CRIT[int(r["n"])] for c in cols))
                          for _, r in rho_table.iterrows()],
    "useful sign (rho<0) / 5": [int(sum(r[c] < 0 for c in cols)) for _, r in rho_table.iterrows()],
})
display(summary.round(3))

# The whole section in four numbers, against the chance rate stated before the result was seen.
n_tests = len(rho_table) * len(cols)
pd.DataFrame({"value": {
    "tests (cells x columns)": n_tests,
    "significant at each row's own p<0.05": int(summary["n_significant / 5"].sum()),
    "expected by chance at p=0.05": round(0.05 * n_tests, 1),
    "useful sign (rho<0)": f"{int(summary['useful sign (rho<0) / 5'].sum())} / {n_tests}",
    "coin flip would give": f"{n_tests // 2} / {n_tests}",
}})

,,mean |rho|,max |rho|,n_significant / 5,useful sign (rho<0) / 5
cell,pooling,,,,
hann w=0.30 (incumbent),mean,0.141,0.301,0,2
hann w=0.10,mean,0.406,0.886,1,2
hann w=0.025,mean,0.349,0.829,0,2
dpss_impulse w=0.30,mean,0.451,0.771,0,2
dpss_impulse w=0.20,mean,0.440,0.657,0,4
dpss_impulse w=0.10,mean,0.371,0.771,0,1
dpss_impulse w=0.05,mean,0.291,0.543,0,4
dpss_impulse w=0.03,mean,0.189,0.314,0,2
dpss_impulse w=0.025,mean,0.215,0.832,1,4


,value
tests (cells x columns),70
significant at each row's own p<0.05,5
expected by chance at p=0.05,3.5
useful sign (rho<0),40 / 70
coin flip would give,35 / 70


## 3. C3b — four cells' µ against the engineered features, under three readouts

Sections 1–2 score µ **on its own**, on four tasks. This section asks the question the paper actually
has to answer: does that µ add anything to the 25 engineered features once the baseline is allowed a
**nonlinear** readout? Roadmap row **C3b**, approved 2026-08-25 as a *control* (`--families gbm mlp`),
reported beside the linear headline and never replacing it.

**Four cells, and how they were chosen** — the union of two clauses, fixed before the table was read:

| cell | mean4 rank (section 2's `mean` scores) | in via |
|---|---|---|
| `dpss_impulse w=0.03 fb0.01` | 1 (0.5783) | both clauses |
| `hann w=0.10` | 2 (0.5768) | top-3 only |
| `dpss_impulse w=0.025 fb0.01` | 3 (0.5746) | both clauses |
| `hann w=0.30` (incumbent) | 5 (0.5712) | named only |

The published C3b table carried the incumbent alone; the other three were extracted and scored for this
section (18 arms × two star populations, then F1 over 25 arms).

**Three arms × three readouts, all at pooling `mean` (N1 headline readout), 6 encoder seeds:**

```
  features          the 25 engineered features        (seedless under `linear`; arm-independent, so
                                                       its column is identical in all four blocks)
  mu                z=128 mean-pooled mu, this cell   (6 seeds)
  features (+) mu   the two concatenated              (6 seeds)

  linear   logistic / ridge          <- the F1 headline readout
  gbm      gradient boosting         <- C3 / C3b, 6 random_state seeds
  mlp      multi-layer perceptron    <- C3 / C3b, 6 random_state seeds
```

**Checkpoint — one rule for all four cells, and it is not the rule sections 1–2 use.** Every arm here
reads `best_recon_aux`, because `exp07_hann0p3_fbwd` seeds 0–5 **predate** `best_recon_only.pt` and have
none; it is the only checkpoint the four cells share, and every published F1/C3b number rides it. That
sits against A3 (an aux-swept cell should be selected by an aux-free metric) and **section 4 resolves
the same tension the other way** for the two `fb0.01` cells. Because it does, the disagreement is
measurable rather than arguable, and the first table below reports it: over the 40 linear rows the two
checkpoints share, mean |Δ| **0.0007–0.0008** and max **0.0030**, while the arm-independent `features`
column agrees to **exactly 0.0** — a cross-file control on two independently produced artifacts.

**Tasks: the 10 reportable ones, `flare` dropped.** The probe menu is 7 (ADR-0010) plus the 4 v1 tasks;
`flare` is excluded from v1 metrics (ADR-0001) and is carried on the menu only as an explicit null, so
it is not a result in either direction and is left out here at the user's request. The four v1 tasks are
**tinted** in both tables.

**Three metrics live in these tables.** `pr_auc` for the detection probes, `roc_auc` for `rgb_vs_heb`,
`r2` for `numax_hon` and `rotation_period`. Compare **along a row, never down a column** — the reverse
of section 1's rule, because here it is the rows that share a metric and a prevalence.

In [6]:
F1 = EXP / "f1_fusion_scorecard"        # linear arms, the published F1 headline
C3B = EXP / "f1_nonlinear_control"      # gbm / mlp arms, the published C3b control
F1X = EXP / "f1_exp09_cells"            # 2026-08-29: both of the above, re-run over four exp09 cells
MENU_ONLY = EXP / "exp09_menu_mu" / "f1_style"   # section 4's best_recon_only arms; read here only as a control

FAMS = ["linear", "gbm", "mlp"]
ARMS = {"features_only": "features", "mu": "mu", "features_plus_mu": "features (+) mu"}
V1 = ["pulsating", "eb", "rotation", "transit"]                       # the 4 highlighted tasks
MENU = ["osc_giant", "solar_like_osc", "ijspeert", "rgb_vs_heb", "numax_hon", "rotation_period"]
TASK_ORDER = V1 + MENU                                                # flare deliberately absent
EXACT = 1e-12   # "the same number", not "close": one row differs by 8.3e-17, i.e. float round-off
# Union of {top 3 on mean4 at `mean`} and {w0p03 fb0.01, w0p025 fb0.01, hann0p3_fbwd}: hann w=0.10 is in
# only via the top-3 clause (mean4 rank 2), the incumbent only via the named clause (rank 5).
SCORED = {"hann  w=0.30 (incumbent)": "hann0p3_fbwd",
          "hann  w=0.10": "exp09_hann_w0p10",
          "dpss_impulse  w=0.03  fb0.01": "exp09_dpss_impulse_w0p03_fb0p01",
          "dpss_impulse  w=0.025  fb0.01": "exp09_dpss_impulse_w0p025_fb0p01"}

# readout == 'mean' is the reported probe score (decision 2026-08-29). `mean_std` and `mean_perp_amp`
# rows exist in f1_absolute.csv and are deliberately not read here.
absolute = pd.read_csv(F1X / "f1_absolute.csv").query(
    "readout == 'mean' and reportable and task != 'flare' and arm_set in @ARMS")
assert sorted(absolute.task.unique()) == sorted(TASK_ORDER), sorted(absolute.task.unique())
assert absolute.groupby("task")[["metric", "n_test"]].nunique().eq(1).all().all()
assert set(absolute.query("family in @SCORED.values()").n_seeds) == {6}

# Footing 1 (rule 8): the re-run must reproduce the PUBLISHED incumbent and features numbers, or the four
# cells are not on one measurement. 90 rows = 10 tasks x 3 readouts x (1 features + 2 encoder arms). The
# published artifacts are read only for this check.
_pub = pd.concat([pd.read_csv(F1 / "f1_absolute.csv"), pd.read_csv(C3B / "f1_absolute.csv")],
                 ignore_index=True)
_key = ["task", "readout", "readout_family", "arm_set", "family", "metric"]
_m = (_pub.query("readout == 'mean' and family in ['features', 'hann0p3_fbwd'] and arm_set in @ARMS")
      .merge(absolute, on=_key, suffixes=("_pub", "_new")))
assert len(_m) == 90 and (_m.score_mean_pub - _m.score_mean_new).abs().max() < EXACT, len(_m)

# Footing 2 -- the checkpoint confound, measured on THIS table rather than cited. Every arm here reads
# `best_recon_aux`, because exp07_hann0p3_fbwd seeds 0-5 predate best_recon_only.pt and have none, so it
# is the only checkpoint all four cells share. That runs against A3 (aux-swept cells should be selected
# by an aux-free metric), and section 4 resolves the same tension the other way. Section 4's artifact
# therefore holds the two fb0.01 cells at `best_recon_only`, which makes the disagreement measurable.
# `readout_family` is IN the merge key: section 4's artifact is linear-only, so leaving it out silently
# compares this table's GBM and MLP rows against linear ones.
_only = pd.read_csv(MENU_ONLY / "f1_absolute.csv").query(
    "readout == 'mean' and readout_family == 'linear' and reportable and task != 'flare' "
    "and arm_set in @ARMS")
_c = absolute.merge(_only, on=["task", "readout_family", "arm_set", "family"],
                    suffixes=("_aux", "_only"))
# 50 = 10 tasks x (2 fb0.01 cells x {mu, fusion} + the 1 shared features column)
assert len(_c) == 5 * len(TASK_ORDER), len(_c)
ckpt_confound = (_c.assign(d=(_c.score_mean_aux - _c.score_mean_only).abs())
                 .groupby("family").d.agg(n="size", mean_abs="mean", max_abs="max"))
assert ckpt_confound.loc["features", "max_abs"] < EXACT   # arm-independent: two runs must agree exactly
assert ckpt_confound.max_abs.max() < 0.005                # and the encoder arms must stay small

frames = []
for label, fam in SCORED.items():
    wide = (absolute.query("family in [@fam, 'features']")
            .pivot_table(index="task", columns=["readout_family", "arm_set"], values="score_mean")
            .reindex(TASK_ORDER)[[(f, a) for f in FAMS for a in ARMS]])
    frames.append(wide.assign(cell=label).set_index("cell", append=True))
c3b_abs = pd.concat(frames)
c3b_abs.columns = pd.MultiIndex.from_tuples([(f, ARMS[a]) for f, a in c3b_abs.columns],
                                            names=["readout (pooling = mean)", "arm"])
_metric = absolute.drop_duplicates("task").set_index("task").metric
c3b_abs = (c3b_abs.assign(metric=[_metric[t] for t, _ in c3b_abs.index]).set_index("metric", append=True)
           .reorder_levels(["task", "metric", "cell"]).loc[TASK_ORDER])


def highlight_best_arm(row: pd.Series) -> pd.Series:
    """Bold the winning arm inside each readout family; tint the four v1 tasks."""
    base = "background-color: #eef3fb; " if row.name[0] in V1 else ""
    style = pd.Series(base, index=row.index)
    for fam in FAMS:
        best = max([(fam, a) for a in ARMS.values()], key=lambda c: row[c])
        style[best] = base + "font-weight: bold; background-color: #cdeccd"
    return style


display(ckpt_confound.round(4))
c3b_abs.style.apply(highlight_best_arm, axis=1).format("{:.3f}")

,n,mean_abs,max_abs
family,,,
exp09_dpss_impulse_w0p025_fb0p01,20,0.0007,0.0027
exp09_dpss_impulse_w0p03_fb0p01,20,0.0008,0.0030
features,10,0.0000,0.0000


In [7]:
# Which CELL tops each arm, tallied over the 10 tasks. `features` is excluded: it is arm-independent and
# identical in all four blocks, so an argmax over cells there would be reporting float tie-breaking.
_feat = c3b_abs.xs("features", level="arm", axis=1)
assert _feat.groupby(level=["task", "metric"]).transform(lambda s: s.max() - s.min()).max().max() == 0.0

tally = pd.DataFrame({(fam, arm): (c3b_abs[(fam, arm)].unstack("cell")[list(SCORED)]
                                   .idxmax(axis=1).value_counts().reindex(SCORED, fill_value=0))
                      for fam in FAMS for arm in ["mu", "features (+) mu"]}).T
tally.index.names = ["readout", "arm"]
tally.columns.name = "cell tops N of 10 tasks"
tally

cell tops N of 10 tasks  hann  w=0.30 (incumbent)  hann  w=0.10  \
readout arm                                                       
linear  mu                                      2             0   
        features (+) mu                         5             0   
gbm     mu                                      1             5   
        features (+) mu                         2             3   
mlp     mu                                      0             2   
        features (+) mu                         1             1   

cell tops N of 10 tasks  dpss_impulse  w=0.03  fb0.01  \
readout arm                                             
linear  mu                                          2   
        features (+) mu                             0   
gbm     mu                                          3   
        features (+) mu                             3   
mlp     mu                                          7   
        features (+) mu                             2   

cell tops N of 10 tasks  dpss_impulse  w=0.025  fb0.01  
readout arm                                             
linear  mu                                           6  
        features (+) mu                              5  
gbm     mu                                           1  
        features (+) mu                              2  
mlp     mu                                           1  
        features (+) mu                              6

### What 3a shows, and what it does not

**Shows (1) — the readout decides which arm wins, and that holds in every one of the four cells.**
Counts are the argmax arm over the 10 tasks:

```
  cell                        linear                 gbm                  mlp
                        fus  feat  mu        fus  feat  mu        fus  feat  mu
  hann  w=0.30            6    3    1          5    5    0          1    9    0
  hann  w=0.10            6    3    1          6    4    0          0    9    1
  dpss  w=0.03  fb0.01    6    2    2          6    4    0          1    7    2
  dpss  w=0.025 fb0.01    6    2    2          6    4    0          0    8    2
```

`features ⊕ µ` tops **6 of 10 under `linear` in all four cells**; `features` alone tops 7–9 of 10 under
`mlp` in all four. Swapping the encoder does not move this — it is a readout effect, exactly as the
published single-cell table said, and now with the cell axis varied instead of assumed.

**Shows (2) — which *cell* is best is not a well-posed question without naming the readout.** The tally
cell above counts how often each cell tops each arm, and the answer changes completely across readouts:

```
  arm      linear                    gbm                       mlp
  mu       w0.025 fb 6/10            hann w=0.10 5/10          w0.03 fb 7/10
  fusion   incumbent + w0.025 fb     spread 3/3/2/2            w0.025 fb 6/10
           5/10 each
```

Section 2's `mean4` ranking put `w0.03 fb0.01` first on the four v1 tasks; on these ten tasks under a
linear readout the µ arm is topped most often by `w0.025 fb0.01`, and under MLP by `w0.03 fb0.01`. Three
different orderings from three readouts over the same four encoders.

**Shows (3) — µ alone still loses to the features, in every cell.** `µ − features` is positive on 3–4
of 10 tasks under `linear` and on 0–3 under the nonlinear readouts, for all four cells. The floored
cells buy a little of this back under `mlp` (2 of 10 each, against 0 for the incumbent), which is the
only place the cell axis moves the qualitative picture at all.

**Does not show.** These are **argmax counts, not tests.** No error bar appears in this table, and many
of the margins behind those counts are far smaller than the seed spread — a 6/10 and a 5/10 in the same
column may be the same picture twice. Section 3b's deltas carry the 1·SE / 2·SE for the incumbent; the
other three cells have no scored comparison anywhere in this notebook, and the cross-cell differences
above are not evidence of anything until one is computed. The absolute scores are also not comparable
**down** a column: three metrics and ten different prevalences.

One more thing this cannot separate, and it is the whole reason 3b exists: comparing the `linear`
fusion column against the `gbm` features column changes the readout **and** the input at once.

### 3b. The fusion delta — what µ adds on top of the features, per readout

Same arms, differenced. This is the C3b verdict table.

```
  delta(task, readout) = score(features (+) mu) - score(features)     paired per seed
  1*SE  = sd over the 6 seeds / sqrt(6)          <- printed per task and per readout
  2*SE  = the project's reporting bar
```

**Scope: the incumbent only.** 3a carries four cells; this table deliberately does not. It is the
published C3b verdict, read from the published artifacts, and widening it would mean adding 30 more
rows of deltas that no pre-registration asks about — while the interesting comparison for the other
three cells (cell A's fusion against cell B's) is a *different* statistic that nothing here computes.
The rows for the other three exist in `f1_exp09_cells/f1_summary.csv` if that question is ever asked.

**Both thresholds are printed for every task**, not just the bar, because most of these deltas are small
and a reader cannot tell a 1.9x margin from a 0.4x one without the denominator. Shading of the delta:

```
  bold green   delta >= +2*SE        pale green   +1*SE <= delta < +2*SE
  bold red     delta <= -2*SE        pale red     -2*SE <  delta <= -1*SE
  no shade     |delta| < 1*SE        <- inside one standard error of zero
```

**Paired per seed matters here and was a bug once.** Under `linear` the feature arm is deterministic and
scored once; under GBM/MLP it carries its own `random_state` seed, and F1's first pass differenced 6
fusion seeds against a single features fit — an offset of up to 0.0062 against margins the same size.
The delta is now paired, and the cross-script check against C3 reads exactly 0.0 on both families.

**Untrained delta is the control column.** Same 128 columns, untrained encoder. If a positive fusion
delta is just "more columns help a hungry readout", the untrained arm gains too; where the untrained arm
loses and the trained one gains, the addition is not generic. It is a **single fit**, so it has no error
bar and none is invented for it.

Rows are sorted by the GBM delta, the column the control was built to interrogate. The four v1 tasks are
tinted.

In [8]:
deltas = (pd.concat([pd.read_csv(F1 / "f1_summary.csv"), pd.read_csv(C3B / "f1_summary.csv")],
                    ignore_index=True)
          # readout == 'mean' is the reported probe score (decision 2026-08-29), not a filter of
          # convenience: `mean_std` rows exist in f1_summary and are deliberately not read here.
          .query("readout == 'mean' and reportable and task != 'flare' "
                 "and contrast == 'fusion_minus_features' and family in ['hann0p3_fbwd', 'untrained']"))
assert sorted(deltas.query("family == 'hann0p3_fbwd'").task.unique()) == sorted(TASK_ORDER)
assert set(deltas.readout) == {"mean"}

trained = deltas.query("family == 'hann0p3_fbwd'").copy()
untrained = deltas.query("family == 'untrained'")
# 1*SE from the seed spread itself, not by halving the published 2*SE -- the two must agree, so check.
trained["delta_1se"] = trained.delta_sd / np.sqrt(trained.n_seeds)
assert np.allclose(2 * trained.delta_1se, trained.delta_2se)

piv = lambda d, v: d.pivot(index="task", columns="readout_family", values=v).reindex(TASK_ORDER)[FAMS]
c3b_delta = pd.concat({fam: pd.DataFrame({"fusion delta": piv(trained, "delta_mean")[fam],
                                          "1*SE": piv(trained, "delta_1se")[fam],
                                          "2*SE": piv(trained, "delta_2se")[fam],
                                          "untrained delta": piv(untrained, "delta_mean")[fam]})
                       for fam in FAMS}, axis=1)
c3b_delta = c3b_delta[FAMS].sort_values(("gbm", "fusion delta"), ascending=False)
c3b_delta.columns.names = ["readout (pooling = mean)", ""]


def highlight_survivors(row: pd.Series) -> pd.Series:
    """Green + bold past 2*SE, pale green past 1*SE, red past -1*SE; tint the four v1 tasks."""
    base = "background-color: #eef3fb; " if row.name in V1 else ""
    style = pd.Series(base, index=row.index)
    for fam in FAMS:
        d, se = row[(fam, "fusion delta")], row[(fam, "1*SE")]
        if d >= 2 * se:
            style[(fam, "fusion delta")] = "font-weight: bold; background-color: #a8d8a8"
        elif d >= se:
            style[(fam, "fusion delta")] = "background-color: #dff0df"
        elif d <= -2 * se:
            style[(fam, "fusion delta")] = "font-weight: bold; background-color: #eda8a8"
        elif d <= -se:
            style[(fam, "fusion delta")] = "background-color: #f7dede"
    return style


c3b_delta.style.apply(highlight_survivors, axis=1).format("{:+.3f}")

### What 3b shows, and what it does not

**Shows — the pre-registered risk fired, and `transit` is the exception.**

```
  linear   6 of 10 past +2*SE   (transit, eb, rotation_period, numax_hon, pulsating, solar_like_osc)
           1 more past +1*SE    (rotation, +0.016 at 1.9x SE -- the bar it misses is 2*SE, not zero)
           3 past -2*SE         (rgb_vs_heb, osc_giant, ijspeert)
  gbm      4 of 10 past +2*SE   (transit, eb, rgb_vs_heb, rotation_period)
           0 in the 1-2*SE band -- GBM's seed spread is so tight that nothing lands there
  mlp      0 of 10 past +2*SE;  7 past -2*SE
```

`transit` is the only task where **µ's contribution grows** when the baseline is given a stronger
readout: +0.027 → **+0.041**, the largest GBM delta in the table, and its untrained control is +0.014,
a third the size. That is mechanistically coherent — transit is the localized task, where global
engineered features cannot represent the signal no matter how much capacity the readout has, and it is
the same asymmetry the MIL work kept finding (pooling gains are transit-only, ADR-0008-lite).

**All four GBM survivors beat their untrained control** (untrained: +0.014, −0.003, −0.012, −0.032), so
what survives is not generic column addition. The MLP column shows the opposite signature: the untrained
arm loses by *more* on almost every probe (−0.10 to −0.23), which is 128 collinear columns diluting a fit
rather than µ misleading it.

**Does not show.** This is one encoder (`hann w=0.30`), one pooling (`mean`), and one feature basis (the
25). It is not a claim about SSL in general, and it says nothing about the exp09 cells in sections 1–2 —
none of them has been through the fusion scorecard. It also carries no supervised-CNN arm: rows **C1/C2**
are what would tell you whether a trained-from-scratch classifier reaches these numbers directly.

**The claim this licenses**, unchanged from `f1_fusion_scorecard_README.md` §C3b and repeated here so the
table is not read looser than it was written:

> µ adds to a **linear** readout on engineered features across most of the 10 tasks; under a nonlinear
> readout on the same features that contribution largely disappears, except on the localized `transit`
> task, where it survives and grows.

## 4. C1 / C2 — the supervised baselines, beside three cells' µ

Sections 1–3 all score a **frozen encoder's µ**. This section adds the two arms that have no encoder at
all: models trained end-to-end on labels, roadmap rows **C1** (D20, the raw-CNN baseline Yue Ma made
mandatory in W6, merged with the Y13c supervised ceiling) and **C2** (Y13b, the independent
model-family check). They are **external baselines** under ADR-0012 decision 3 — reported beside the
probe, never as it.

C1 is what `swm/eval/new_task_ceiling.py:12` calls the deliberately-absent **Ceiling B**: it answers
"how much signal do the data and labels contain at all?", which no arm above can answer.

```
  C1 conv   Encoder's Conv-BN-ReLU-MaxPool stack, VERBATIM -> fc_mu 4096->128 -> dropout -> Linear(128,1)
  C2 mlp    Linear(256,256)-ReLU-Linear(256,256)-ReLU      -> fc_mu  256->128 -> dropout -> Linear(128,1)
                                                              ^^^^^ the ONLY difference is the trunk

  star score = mean over the star's window outputs, computed IN-GRAPH
               -> the training loss and the reported metric are the same quantity
```

### Read the three tables as one comparison with one thing varying

**C1 and C2 do not depend on the encoder.** They are trained from scratch, so their two columns are
**identical in all three tables** — that is not a copy-paste error, it is the point. Only
`features`, `µ` and `features ⊕ µ` change between tables, so each table asks:

```
  does THIS cell's frozen µ, read linearly, reach what a supervised model of the same
  architecture reaches on the same labels and the same input?
```

`features` is arm-independent too and is therefore also identical across the three; it is repeated so
each table can be read on its own (rule 5: a fusion number is never quoted without the µ-only and
features-only arms beside it).

### What is being compared, and on what

| arm | what it is | seeds |
|---|---|---|
| `features` | the 25 engineered features, logistic/ridge | 1 (deterministic) |
| `µ` | z=128 mean-pooled µ from **that row's cell**, frozen encoder | 6 |
| `features ⊕ µ` | the two concatenated, same linear readout | 6 |
| **`C1 conv`** | supervised Conv1D trained end-to-end **on this task** | 3 |
| **`C2 mlp`** | supervised dense net, same protocol, no convolution | 3 |

**Tasks: the 10 reportable ones, `flare` dropped**, same set and same order as section 3. The four v1
tasks are **tinted blue**.

### The highlighting rule, stated before the numbers

Marking the row maximum alone would be misleading here: on most rows the top two arms sit inside each
other's noise, and a bold winner reads as a result when it is a coin flip. So the top arm is tested
against the **runner-up**, not against zero:

```
  gap      = score(top arm) - score(runner-up)
  combined = sqrt( 2SE(top)^2 + 2SE(runner-up)^2 )      # the seedless features arm contributes 0 (F1's convention)

  bold green   gap >  combined    ->  the top arm is separated from the next best
  pale green   gap <= combined    ->  it is only the argmax; do not read it as a win
```

This is the same discipline section 3b applies to its deltas, moved onto absolute scores. Expect most
rows to be pale.

### Three things that must be said before the numbers

1. **The checkpoint rule differs by cell, deliberately.** `hann w=0.30` is read at `best_recon_aux` —
   it is the published reference every F1 number rides. The two `fb0.01` cells are read at
   `best_recon_only`, which is not optional for them: their sweep axis **is** the aux term, so an
   aux-bearing selector would compare them at a checkpoint chosen by a different rule (ADR-0012 A3,
   and the same rule sections 1–2 use for these cells). ADR-0012 measured this confound directly at
   mean |Δ| 0.0006, max 0.0047.
2. **The menu µ for the two `fb0.01` cells did not exist before this section** and was extracted for
   it: 12 arms over the 22,860-star pool plus the v1 subset, into their own cache dir (the exp09
   µ-cache trap: caches key on `{arm}.npz` with no checkpoint in the key). Two footing checks, both run
   before any score was read — the new subset cache re-pooled per star reproduces `exp09_forensics`'
   already-pooled µ to **max 4.0e-4** over 24 split-comparisons (the magnitude F1 records for two encode
   passes of one checkpoint), and the pool tics and counts match F1's cache **exactly**. A third runs
   inside the next cell: the engineered arm is arm-independent, so the two independently produced
   `f1_absolute.csv` files must agree on it to **0.0**.
3. **Supervised seeds and encoder seeds are different things** and are never paired: a supervised seed
   is an init/shuffle seed, an encoder seed is a pretraining seed. No delta between a supervised column
   and a µ column is computed here — the columns are placed side by side, and the only inference drawn
   is the separated/not-separated call above.

Three metrics live in these tables (`pr_auc`, `roc_auc` for `rgb_vs_heb`, `r2` for the two regressions),
so compare **along a row, never down a column** — section 3's rule, unchanged.

In [9]:
C1C2 = EXP / "c1c2_supervised"                    # roadmap C1/C2, 66 supervised runs
MENU09 = EXP / "exp09_menu_mu" / "f1_style"       # menu+v1 mu for the two fb0.01 cells, extracted for this section

S4_CELLS = {"hann  w=0.30 (incumbent)": "hann0p3_fbwd",
            "dpss_impulse  w=0.03  fb0.01": "exp09_dpss_impulse_w0p03_fb0p01",
            "dpss_impulse  w=0.025  fb0.01": "exp09_dpss_impulse_w0p025_fb0p01"}
SUP_ARMS = {"conv_supervised": "C1 conv", "mlp_raw": "C2 mlp"}

mu_abs = pd.concat([pd.read_csv(F1 / "f1_absolute.csv"), pd.read_csv(MENU09 / "f1_absolute.csv")],
                   ignore_index=True)
mu_abs = mu_abs[(mu_abs.readout == "mean") & (mu_abs.readout_family == "linear") & mu_abs.reportable
                & (mu_abs.task != "flare") & mu_abs.arm_set.isin(ARMS)]
sup_abs = pd.read_csv(C1C2 / "c1c2_absolute.csv").query("task != 'flare'")

# Footing, all three before any table renders (rule 8). (a) the incumbent still reproduces F1's
# published 6-seed eb number; (b) the newly extracted cells carry the six seeds F1 scores hann at, and
# the supervised arms their three; (c) CROSS-FILE control -- the engineered arm is arm-independent, so
# the two f1_absolute.csv files must agree on it exactly, and averaging them silently would hide drift.
_eb = mu_abs.query("family == 'hann0p3_fbwd' and arm_set == 'mu' and task == 'eb'").score_mean.iloc[0]
assert abs(_eb - 0.7710) < 5e-4, _eb
assert set(mu_abs[mu_abs.family.isin(S4_CELLS.values())].n_seeds) == {6}
assert set(sup_abs.n_seeds) == {3}
_feat_spread = mu_abs.query("arm_set == 'features_only'").groupby("task").score_mean.agg(np.ptp)
assert _feat_spread.max() < 1e-9, _feat_spread
mu_abs = mu_abs.drop_duplicates(["task", "family", "arm_set"])


def cell_table(family: str, value: str) -> pd.DataFrame:
    """One cell's five arms on the 10 reportable tasks; `value` picks score_mean or score_2se.

    The two supervised columns are encoder-independent and are therefore joined in per TASK, not per
    cell -- which makes their repetition across the three tables structural rather than a coincidence
    the reader has to verify.
    """
    rows = mu_abs[mu_abs.family.isin([family, "features"])]
    table = rows.pivot(index="task", columns="arm_set", values=value).rename(columns=ARMS)
    table = table[list(ARMS.values())]
    for arm, label in SUP_ARMS.items():
        table[label] = sup_abs[sup_abs.arm == arm].set_index("task")[value]
    table = table.reindex(TASK_ORDER)
    assert table.notna().all().all(), table[table.isna().any(axis=1)]
    return table


def style_cell(family: str) -> "pd.io.formats.style.Styler":
    """Render one cell's table, distinguishing a SEPARATED winner from a mere argmax.

    Highlighting the row maximum alone would read as "this arm wins" on rows where the top two arms sit
    inside each other's noise -- which is most of them here. So the top arm is compared against the
    runner-up at their combined 2*SE (the engineered arm is seedless and contributes 0, F1's convention):

      bold green   top arm separated from the runner-up   |top1 - top2| >  sqrt(se1^2 + se2^2)
      pale green   top arm is only the argmax             |top1 - top2| <= that
    """
    scores, errs = cell_table(family, "score_mean"), cell_table(family, "score_2se")

    def paint(row: pd.Series) -> pd.Series:
        base = "background-color: #eef3fb; " if row.name in V1 else ""
        style = pd.Series(base, index=row.index)
        top, second = row.nlargest(2).index
        gap = row[top] - row[second]
        combined = np.hypot(errs.loc[row.name, top], errs.loc[row.name, second])
        if gap > combined:
            style[top] = base + "font-weight: bold; background-color: #a8d8a8"
        else:
            style[top] = base + "background-color: #dff0df"
        return style

    metric = mu_abs.drop_duplicates("task").set_index("task").metric.reindex(TASK_ORDER)
    shown = scores.copy()
    shown.index = pd.MultiIndex.from_arrays([scores.index, metric], names=["task", "metric"])
    label = [k for k, v in S4_CELLS.items() if v == family][0]
    # paint reads the flat-index row (V1 membership, matching errs row); display carries (task, metric)
    return (shown.style.apply(lambda r: paint(scores.loc[r.name[0]]), axis=1).format("{:.3f}")
            .set_caption(f"<b>{label}</b> &nbsp;&nbsp; mu arms 6 seeds, C1/C2 3 seeds, readout `mean`, "
                         f"linear &nbsp;|&nbsp; <b>bold</b> = top arm separated from runner-up at "
                         f"combined 2*SE, pale = argmax only &nbsp;|&nbsp; C1/C2 are "
                         f"encoder-independent and repeat"))


for family in S4_CELLS.values():
    display(style_cell(family))

,arm_set,features,mu,features (+) mu,C1 conv,C2 mlp
task,metric,,,,,
pulsating,pr_auc,0.789,0.806,0.846,0.816,0.756
eb,pr_auc,0.742,0.771,0.775,0.774,0.720
rotation,pr_auc,0.540,0.559,0.555,0.571,0.535
transit,pr_auc,0.190,0.144,0.218,0.133,0.108
osc_giant,pr_auc,0.920,0.854,0.918,0.873,0.885
solar_like_osc,pr_auc,0.320,0.336,0.390,0.395,0.278
ijspeert,pr_auc,0.508,0.440,0.454,0.457,0.456
rgb_vs_heb,roc_auc,0.758,0.660,0.708,0.522,0.761
numax_hon,r2,0.831,0.802,0.867,0.841,0.834


,arm_set,features,mu,features (+) mu,C1 conv,C2 mlp
task,metric,,,,,
pulsating,pr_auc,0.789,0.802,0.840,0.816,0.756
eb,pr_auc,0.742,0.787,0.781,0.774,0.720
rotation,pr_auc,0.540,0.573,0.567,0.571,0.535
transit,pr_auc,0.190,0.152,0.216,0.133,0.108
osc_giant,pr_auc,0.920,0.868,0.921,0.873,0.885
solar_like_osc,pr_auc,0.320,0.320,0.382,0.395,0.278
ijspeert,pr_auc,0.508,0.437,0.435,0.457,0.456
rgb_vs_heb,roc_auc,0.758,0.662,0.707,0.522,0.761
numax_hon,r2,0.831,0.804,0.867,0.841,0.834


,arm_set,features,mu,features (+) mu,C1 conv,C2 mlp
task,metric,,,,,
pulsating,pr_auc,0.789,0.792,0.837,0.816,0.756
eb,pr_auc,0.742,0.790,0.787,0.774,0.720
rotation,pr_auc,0.540,0.583,0.583,0.571,0.535
transit,pr_auc,0.190,0.141,0.204,0.133,0.108
osc_giant,pr_auc,0.920,0.866,0.921,0.873,0.885
solar_like_osc,pr_auc,0.320,0.314,0.376,0.395,0.278
ijspeert,pr_auc,0.508,0.445,0.449,0.457,0.456
rgb_vs_heb,roc_auc,0.758,0.667,0.709,0.522,0.761
numax_hon,r2,0.831,0.806,0.868,0.841,0.834


### What section 4 shows, and what it does not

**Shows — the supervised arms do not take the table over.** Across the 30 (cell × task) rows, C1 or C2
is the argmax **7 times**, and survives the separation test **twice** — both of them `solar_like_osc`,
in the two `fb0.01` cells:

```
  argmax per cell (before the separation test)      separated (bold) rows
  cell                 fusion  mu  features  C1  C2        of 10
  hann  w=0.30            5     0      2      2   1          6
  dpss  w=0.03  fb0.01    5     2      1      1   1          6
  dpss  w=0.025 fb0.01    5     2      1      1   1          5
```

`features ⊕ µ` is the argmax on **5 of 10 in every cell**, and is the separated top on `transit`,
`numax_hon` and `rotation_period` in all three (plus `pulsating` in the incumbent). In the incumbent's
table **neither supervised arm is a separated winner anywhere** — C1's two argmaxes there (`rotation`
+0.011 against 0.012, `solar_like_osc` +0.005 against 0.007) both sit inside the combined noise.

**Two rows deserve naming because they are the exceptions.**

```
  ijspeert     features is the separated top in ALL THREE cells, +0.052 over C1 at combined 2SE 0.003
               -> the one task where the engineered basis beats everything, decisively and repeatably
  solar_like_osc  C1 is the separated top in both fb0.01 cells (+0.013, +0.019) but NOT in the incumbent
               -> C1 does not move between tables; what moved is the fusion arm, 0.390 -> 0.382 -> 0.376
```

**`rgb_vs_heb` is where the C1/C2 contrast is real and the table's top is not.** C2 is the argmax
(0.761) but is **not** separated from `features` (0.758; gap 0.0025 vs combined 0.0056) — so nothing
here licenses "the MLP wins this task". What *is* enormous is C2 over **C1**: 0.761 vs 0.522, a gap of
0.239 against a combined 2·SE of 0.006. Those are two different comparisons and only the second is a
result: on 755 training stars the 132 k-parameter dense trunk beats the 1.1 M-parameter conv trunk,
which is a capacity story about the supervised arms, not a claim about the task's best model.

**The three cells barely differ, and where they do it corroborates wave 7.** The µ column moves most on
`rotation` (0.559 → 0.573 → 0.583, rising as the floor is added) and `pulsating` (0.806 → 0.802 →
0.792, falling). The pulsating direction is the one P12-D predicted — the free-bits floor binds on
pulsating — but **this block does not establish it**: the 2·SE on those three values are 0.015, 0.011
and 0.016, so a 0.014 drop is inside the noise. Direction agrees; separation does not follow.

**Does not show.**

- **No delta is computed between a supervised column and a µ column**, and none should be read off by
  subtracting: a supervised seed is an init/shuffle seed and an encoder seed is a pretraining seed, so
  the pairing that makes section 3b's deltas trustworthy does not exist here.
- **No comparison between the three cells is scored.** The tables are placed side by side, exactly as
  section 1 does, and carry no paired-by-seed delta or SE across cells. Two cells' numbers can differ
  by less than their seed spread, and three of the rows above do.
- **The supervised arms are bounded twice over.** They see the same first-segment windows as µ (the
  fairness constraint that makes the comparison architecture-controlled) and they train on 669–16,002
  labelled stars. So a low C1 number bounds *the labelled ceiling at the probe's input scope with the
  labels that exist*, not the ceiling in general — the claim carried in
  `experiments/c1c2_supervised_README.md` is worded that way and this table does not widen it.
- **The two `fb0.01` cells are read at a different checkpoint than the incumbent** (`best_recon_only`
  vs `best_recon_aux`), for the reason stated above. ADR-0012 measured that confound at mean |Δ| 0.0006
  / max 0.0047, which is smaller than most gaps here but not smaller than all of them.
- **`flare` is absent by request**, and separately it is unprintable until L1's visual pass lands.
